# 怎样识别垃圾短信并分析误报？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

词表大小 9282


{'rows_after_deduplication': 5158, 'baseline': {'accuracy': 0.8751937984496124, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'tfidf_nb': {'accuracy': 0.9751937984496124, 'precision': 1.0, 'recall': 0.8012422360248447, 'f1': 0.8896551724137931}}


## 1. 补全：核算召回率
从混淆矩阵提取 TP 与 FN，复算垃圾召回率。

In [2]:
tn, fp, fn, tp = matrix.ravel()
assert np.isclose(tp/(tp+fn), metrics['tfidf_nb']['recall'])
print('漏掉垃圾短信数', fn)

漏掉垃圾短信数 32


## 2. 修改：查看新短信
输入两条自写短信，输出预测概率；不要把这两条方便样例当成性能评测。

In [3]:
examples = ['Meeting at the library at five, see you there.', 'Congratulations! Claim your cash prize now by calling this number.']
scores = model.predict_proba(examples)[:,1]
assert scores.shape == (2,) and ((scores >= 0) & (scores <= 1)).all()
print(list(zip(examples,scores)))

[('Meeting at the library at five, see you there.', 0.0033586900892438225), ('Congratulations! Claim your cash prize now by calling this number.', 0.8357042581374219)]


## 3. 迁移：用训练子集比较纯词模型
从原训练集再切开发验证集，比较仅 unigram 的模型；不触碰原测试集做选择。

In [4]:
subtrain, subval = train_test_split(train_idx, test_size=0.2, stratify=y.loc[train_idx], random_state=81)
word_model = make_pipeline(TfidfVectorizer(min_df=2),MultinomialNB(alpha=0.5))
word_model.fit(messages.loc[subtrain,'text'],y.loc[subtrain])
scores = classification_scores(y.loc[subval].to_numpy(),word_model.predict(messages.loc[subval,'text']))
assert set(subval).isdisjoint(set(test_idx))
print('训练内部验证结果', scores)

训练内部验证结果 {'accuracy': 0.9819121447028424, 'precision': 0.9880952380952381, 'recall': 0.8645833333333334, 'f1': 0.9222222222222223}
